In [1]:
import MDAnalysis as mda
import numpy as np
from MDAnalysis.analysis import align
import pandas as pd

In [2]:
ls

5000frame_6BBV/
6bbv_1000.prmtop*
6BBV_Entropy/
6BBV_MMPBSA/
6bbv_no_water1.prmtop*
6bbv_no_water.dcd*
6bbv_no_water.prmtop*
6bbv.pdb*
6bbv_rgyr_backbone.dat*
6bbv_rgyr.dat*
6bbv_rmsd1.dat*
6bbv_rmsf_output.dat*
6BBV_S_MMGBSA/
6bbv_surf.dat*
6BBV.vmd*
align_mds.py*
amber_codes*
ANTECHAMBER_AC.AC*
ANTECHAMBER_AC.AC0*
ANTECHAMBER_AM1BCC.AC*
ANTECHAMBER_AM1BCC_PRE.AC*
ANTECHAMBER_BOND_TYPE.AC*
ANTECHAMBER_BOND_TYPE.AC0*
aromatic_distance.dat*
aromatic.tcl*
atom_residue_energy_map.csv*
ATOMTYPE.INF*
atom_wise/
atomwise_interaction_fingerprint.csv*
Check/
cluster_info.dat*
cluster_summary.dat*
comlex_6BBV.pdb*
comp.inpcrd*
complex_strip_water.in*
comp.pdb*
comp.prmtop*
comp_solv.inpcrd*
comp_solv_no_water_filtered.pdb*
comp_solv_no_water.pdb*
comp_solv.pdb*
comp_solv.prmtop*
contact_occupancy.py*
convert_nc_to_dcd.in*
cpptraj.log*
D7D.agr*
density.in*
density.mdcrd*
density.out*
density.rst*
dry.dcd*
dry.prmtop*
dynamic_centroid.tcl*
energies/
energy_decomposition_map.png*
equil.in*
equil.m

In [3]:
u = mda.Universe("6bbv_no_water.prmtop", "6bbv_no_water.dcd")

/home/thsti/miniconda3/envs/mda311/lib/python3.11/site-packages/MDAnalysis/coordinates/DCD.py:171: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


In [4]:
protein_backbone = u.select_atoms("protein and backbone")

aligner = align.AlignTraj(
    u,
    u,
    select="protein and backbone",
    in_memory=True
)
aligner.run()

In [10]:

# =========================
# D7D aromatic rings
# =========================

# C6 ring
c6_ring = u.select_atoms(
    "resname D7D and resid 290 and name N1 C N C3 C2 C1"
)

# C5 ring
c5_ring = u.select_atoms(
    "resname D7D and resid 290 and name C1 C2 C12 C13 N4"
)

# Lys and Arg cationic atoms
cation = u.select_atoms(
    "resname LYS ARG and name NZ NE NH1 NH2"
)

def centroid(atoms):
    return atoms.positions.mean(axis=0)

cutoff = 6.9
total_frames = 0

# Store occupancy counts and distances
res_ring_data = {}

for ts in u.trajectory:

    total_frames += 1

    c6_cent = centroid(c6_ring)
    c5_cent = centroid(c5_ring)

    for res in cation.residues:

        resid_name = f"{res.resname}{res.resid}"

        if resid_name not in res_ring_data:
            res_ring_data[resid_name] = {
                "C6_count": 0,
                "C5_count": 0,
                "C6_dist": [],
                "C5_dist": []
            }

        pos = res.atoms.positions

        d_c6 = np.min(np.linalg.norm(pos - c6_cent, axis=1))
        d_c5 = np.min(np.linalg.norm(pos - c5_cent, axis=1))

        # assign interaction to closest ring
        if d_c6 <= cutoff and d_c6 <= d_c5:
            res_ring_data[resid_name]["C6_count"] += 1
            res_ring_data[resid_name]["C6_dist"].append(d_c6)

        elif d_c5 <= cutoff and d_c5 < d_c6:
            res_ring_data[resid_name]["C5_count"] += 1
            res_ring_data[resid_name]["C5_dist"].append(d_c5)

# =========================
# Create results table
# =========================

results = []

for res, d in res_ring_data.items():

    c6_occ = 100.0 * d["C6_count"] / total_frames
    c5_occ = 100.0 * d["C5_count"] / total_frames

    c6_avg = np.mean(d["C6_dist"]) if d["C6_dist"] else 0
    c5_avg = np.mean(d["C5_dist"]) if d["C5_dist"] else 0

    c6_min = np.min(d["C6_dist"]) if d["C6_dist"] else 0
    c6_max = np.max(d["C6_dist"]) if d["C6_dist"] else 0

    c5_min = np.min(d["C5_dist"]) if d["C5_dist"] else 0
    c5_max = np.max(d["C5_dist"]) if d["C5_dist"] else 0

    results.append([
        res,
        round(c6_occ, 2),
        round(c6_avg, 2),
        round(c6_min, 2),
        round(c6_max, 2),
        round(c5_occ, 2),
        round(c5_avg, 2),
        round(c5_min, 2),
        round(c5_max, 2)
    ])

df = pd.DataFrame(
    results,
    columns=[
        "Residue",
        "C6_Occupancy_%",
        "C6_AvgDist",
        "C6_MinDist",
        "C6_MaxDist",
        "C5_Occupancy_%",
        "C5_AvgDist",
        "C5_MinDist",
        "C5_MaxDist"
    ]
)

# Keep only residues with occupancy >30%
df = df[
    (df["C6_Occupancy_%"] > 30.0) |
    (df["C5_Occupancy_%"] > 30.0)
]

df = df.sort_values(
    by=["C6_Occupancy_%", "C5_Occupancy_%"],
    ascending=False
)

print(df.to_string(index=False))

df.to_csv("D7D_pi_cation_above30.csv", index=False)

Residue  C6_Occupancy_%  C6_AvgDist  C6_MinDist  C6_MaxDist  C5_Occupancy_%  C5_AvgDist  C5_MinDist  C5_MaxDist
  LYS40             0.0           0           0           0           44.56        6.56        3.87         6.9


In [11]:
rxt = u.select_atoms("resname D7D")
print(sorted(set(rxt.names)))

['C', 'C1', 'C10', 'C11', 'C12', 'C13', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'H101', 'H102', 'H103', 'H111', 'H112', 'H113', 'H5_1', 'H5_2', 'H7_1', 'H7_2', 'H8_1', 'H8_2', 'H9_1', 'H9_2', 'HC12', 'HC13', 'HC4', 'HC6', 'HN3', 'HN4', 'HXT', 'N', 'N1', 'N2', 'N3', 'N4', 'O', 'O1', 'S']


In [12]:
lig = u.select_atoms("resname D7D")
print("Number of atoms:", len(lig))

for r in lig.residues:
    print("Resid:", r.resid, "Resname:", r.resname)

Number of atoms: 43
Resid: 290 Resname: D7D


In [13]:
lig = u.select_atoms("resname D7D")
print("Number of atoms:", len(lig))

for r in lig.residues:
    print("Resid:", r.resid, "Resname:", r.resname)

Number of atoms: 43
Resid: 290 Resname: D7D


In [27]:
files = {
    "6WTN": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/jak2/6wtn/aromatic_distance.dat",
    "6BBV": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/jak2/6BBV/aromatic_distance.dat",
    "6WTO": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/jak2/final_6wto/aromatic_distance.dat",
    "6LUD": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/egfr/holo/final_6LUD/aromatic_distance.dat",
    "5AMH": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/cereblon_isoform/Holo/5AMH/aromatic_distance.dat",
    "4V30": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/cereblon_isoform/Holo/4v30/aromatic_distance.dat",
    "6V2F": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/HIV_1_capsid/Holo/6V2F/aromatic_distance.dat",
    "2IOG": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/lab_proteins/Estrogen_receptor/2IOG/aromatic_distance.dat",
    "8ARJ": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/lab_proteins/Anaplastic_lymphoma_kinase/8ARJ/aromatic_distance.dat",
    "7FCG": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/lab_proteins/Fatty_acid/7FCG/05_2_2025/aromatic_distance.dat",
    "8C7X": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/lab_proteins/B_raf/8C7X/aromatic_distance.dat"
}

for pdb, file in files.items():

    print("\n" + "="*60)
    print(pdb)
    print("="*60)

    with open(file, "r") as f:
        lines = f.readlines()

    start = None

    # Look for Occupancy section first
    for i, line in enumerate(lines):
        if "===== Occupancy =====" in line:
            start = i
            break

    # If not found, look for Interaction Statistics section
    if start is None:
        for i, line in enumerate(lines):
            if "===== Interaction Statistics =====" in line:
                start = i
                break
    # If not found, look for OCCUPANCY (%)section
    if start is None:
        for i, line in enumerate(lines):
            if "===== OCCUPANCY (%) =====" in line:
                start = i
                break
                

    if start is not None:
        print("".join(lines[start:]))
    else:
        print("No Occupancy or Interaction Statistics section found.")


6WTN
===== Occupancy =====
TYR89-C6ring Occupancy = 97.404 %
TYR89-C5ring Occupancy = 32.476 %
TYR92-C6ring Occupancy = 0.0 %
TYR92-C5ring Occupancy = 0.0 %

===== Average Interacting Distances =====
Average TYR89-C6ring Distance = 6.045762603015367 Å
Average TYR89-C5ring Distance = 6.7609765588862825 Å
Average TYR92-C6ring Distance = 0 Å
Average TYR92-C5ring Distance = 0 Å

===== Minimum and Maximum Interacting Distances =====
Minimum TYR89-C6ring Distance = 4.341185518722934 Å
Maximum TYR89-C6ring Distance = 6.999314547078236 Å
Minimum TYR89-C5ring Distance = 5.625399280091677 Å
Maximum TYR89-C5ring Distance = 6.999986061240788 Å
Minimum TYR92-C6ring Distance = 9999 Å
Maximum TYR92-C6ring Distance = 0 Å
Minimum TYR92-C5ring Distance = 9999 Å
Maximum TYR92-C5ring Distance = 0 Å


6BBV
===== Occupancy =====
C6ring Occupancy = 99.924 %
C5ring Occupancy = 51.398 %

===== Average Interacting Distances =====
Average TYR89-C6ring Distance = 5.721369444240464 Å
Average TYR89-C5ring Distance

In [28]:
import re

files = {
    "6WTN": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/jak2/6wtn/aromatic_distance.dat",
    "6BBV": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/jak2/6BBV/aromatic_distance.dat",
    "6WTO": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/jak2/final_6wto/aromatic_distance.dat",
    "6LUD": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/egfr/holo/final_6LUD/aromatic_distance.dat",
    "5AMH": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/cereblon_isoform/Holo/5AMH/aromatic_distance.dat",
    "4V30": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/cereblon_isoform/Holo/4v30/aromatic_distance.dat",
    "6V2F": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/HIV_1_capsid/Holo/6V2F/aromatic_distance.dat",
    "2IOG": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/lab_proteins/Estrogen_receptor/2IOG/aromatic_distance.dat",
    "8ARJ": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/lab_proteins/Anaplastic_lymphoma_kinase/8ARJ/aromatic_distance.dat",
    "7FCG": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/lab_proteins/Fatty_acid/7FCG/05_2_2025/aromatic_distance.dat",
    "8C7X": "/media/thsti/6938FCB134D4F6F7/Dr.Meenakshi/fda/md_simulations/lab_proteins/B_raf/8C7X/aromatic_distance.dat"
}

for pdb, file in files.items():

    print("\n" + "="*60)
    print(pdb)
    print("="*60)

    with open(file) as f:
        lines = f.readlines()

    current_interaction = None
    found = False

    for line in lines:

        line = line.strip()

        # Interaction name (e.g. TYR89_C6, PHE32_C5)
        if line and not line.startswith("Occupancy") \
           and not line.startswith("Average") \
           and not line.startswith("Minimum") \
           and not line.startswith("Maximum") \
           and not line.startswith("="):
            current_interaction = line

        # Occupancy line
        m = re.search(r'Occupancy\s*=?\s*([\d.]+)', line)

        if m:
            occ = float(m.group(1))

            if occ > 30:
                found = True
                print(f"{current_interaction:20s} {occ:.2f}%")

    if not found:
        print("No interactions with occupancy > 30%")


6WTN
TYR89-C6ring Occupancy = 97.404 % 97.40%
TYR89-C5ring Occupancy = 32.476 % 32.48%

6BBV
C6ring Occupancy = 99.924 % 99.92%
C5ring Occupancy = 51.398 % 51.40%

6WTO
TYR89-C6 Occupancy = 99.714 % 99.71%
TYR89-C5 Occupancy = 55.66 % 55.66%

6LUD
No interactions with occupancy > 30%

5AMH
TRP62_5_C5           85.10%
TRP62_6_C5           91.86%
TRP68_5_C5           86.32%
TRP68_6_C6           58.63%
TRP68_6_C5           99.82%
TRP82_5_C5           83.06%
TRP82_6_C5           67.97%

4V30
PHE59_C6             68.49%
PHE59_C5             57.49%
TRP61_6_C5           56.93%
TRP67_5_C6           53.39%
TRP67_5_C5           99.20%
TRP67_6_C6           91.87%
TRP67_6_C5           99.96%
TRP81_5_C5           82.34%
TRP81_6_C5           50.09%

6V2F
TYR130_C6            63.58%

2IOG
TRP75_6_C6           96.34%

8ARJ
No interactions with occupancy > 30%

7FCG
PHE17_C5 Occupancy = 51.45 % 51.45%
PHE17_C6 Occupancy = 70.682 % 70.68%

8C7X
TRP83_5_C5           73.83%
TRP83_5_C6           98.38%
PH